# 02 · Entrenamiento LoRA (Colab T4)

Dos configuraciones que difieren en **un solo factor: `target_modules`**. Todo lo demás es igual:
r = 16, alpha = 32, dropout = 0.05, lr = 2e-4 (cosine, warmup 3 %), 1 época, batch efectivo 16, fp16, semilla 2026.

| Config | target_modules |
|---|---|
| **A** | `q_proj, k_proj, v_proj, o_proj` (solo atención) |
| **B** | A + `gate_proj, up_proj, down_proj` (atención + MLP) |

**Hipótesis (escrita antes de entrenar):** B tiene ~2.5× más parámetros entrenables y adapta también las MLP, donde se concentra más el conocimiento "factual" (sintaxis y funciones de MySQL, convenciones del esquema). Esperamos que B logre menor *eval loss* y mayor EX, sobre todo en preguntas difíciles, a costa de ~15–25 % más tiempo por paso. A podría generalizar igual de bien si el cuello de botella es solo el formato de salida.

> Consejo: corran A y B **en paralelo** en dos cuentas de Colab (una por integrante).

In [ ]:
# Obtener el código: subir proyecto_3_deep.zip a /content (panel Archivos) o clonar el repo privado
import os, getpass
REPO = "proyecto_3_deep"
if not os.path.exists(f"/content/{REPO}/solucion.py"):
    if os.path.exists(f"/content/{REPO}.zip"):
        !unzip -q -o /content/{REPO}.zip -d /content/{REPO}
        # si el zip traía la carpeta raíz adentro, subirla un nivel
        if not os.path.exists(f"/content/{REPO}/solucion.py") and os.path.exists(f"/content/{REPO}/{REPO}/solucion.py"):
            !mv /content/{REPO}/{REPO}/* /content/{REPO}/ && rm -rf /content/{REPO}/{REPO}
    else:
        usuario = input("Usuario de GitHub: ")
        token = getpass.getpass("Token de GitHub: ")
        url_repo = input("URL del repo (https://github.com/<owner>/<repo>.git): ").replace("https://", f"https://{usuario}:{token}@")
        !git clone -q {url_repo} /content/{REPO}
%cd /content/{REPO}
!pip install -q -r requirements.txt

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
CONFIG = "A"   # cambiar a "B" para la segunda configuración
!python scripts/entrenar.py --config {CONFIG}

In [ ]:
import json
from IPython.display import Image, display
info = json.load(open(f"resultados/entrenamiento_{CONFIG}.json", encoding="utf-8"))
print({k: v for k, v in info.items() if not k.startswith("curva")})
display(Image(f"resultados/curvas_loss_{CONFIG}.png"))

In [ ]:
# Descargar el adapter y los resultados para subirlos al repo
!zip -qr /content/adapter_{CONFIG}.zip adapter_{CONFIG} resultados/entrenamiento_{CONFIG}.json resultados/curvas_loss_{CONFIG}.png
from google.colab import files
files.download(f"/content/adapter_{CONFIG}.zip")